## 2. Setup: Initialize environment and client

In this step, you import the key libraries that will support the workflow:  

- **`re`**: Python’s regular expression module, which you’ll use to extract snippets of code or structured text from the LLM’s output.  
- **`json`**: Provides functions to read and write JSON, useful for handling structured responses returned by the LLM.  
- **`utils`**: A custom helper module provided for this lab. It includes utility functions to work with the dataset, generate charts, and display results in a clean, readable format.  


In [1]:
import re
import json
import utils

print(utils)


<module 'utils' from '/Users/hasstrupezekiel/Desktop/projects/agents-/ungraded-labs/utils.py'>


In [2]:
df = utils.load_and_prepare_data('coffee_sales.csv')
utils.print_html(df.sample(n=5), title="Random Sample of Coffee Sales Data")

hour_of_day,cash_type,money,coffee_name,Time_of_Day,Weekday,Month_name,Weekdaysort,Monthsort,Date,Time
8,card,35.76,Latte,Morning,Wed,Nov,3,11,2024-11-06,08:11:25.120000
15,card,37.72,Latte,Afternoon,Sun,May,7,5,2024-05-12,15:16:19.815000
18,card,35.76,Cappuccino,Night,Thu,Jan,4,1,2025-01-16,18:15:57.449000
17,card,35.76,Latte,Night,Mon,Feb,1,2,2025-02-03,17:14:18.374000
16,card,35.76,Hot Chocolate,Afternoon,Fri,Feb,5,2,2025-02-21,16:48:58.515000


In [ ]:
def generate_chart_code(instruction: str, model: str, out_path_v1: str) -> str:
    prompt = f"""
    You are a data visualization expert.
    
    Return your answer *strictly* in this format:
    
    <execute_python>
    # valid python code here
    </execute_python>
    
    Do not add any explanations, only the tags and the code. 
    
    The code should create a visualization from a DataFrame 'df' with these columns:
    
    - date (datetime64 - already parsed; use df['date'].dt.year, df['date'].dt.month, etc.)
    - time (string, HH:MM - do NOT concatenate or combine with the date column)
    - cash_type (string: 'card' or 'type')
    - card (string)
    - price (number)
    - coffee_name (string)
    - quarter (int, 1-4, compute from the date) 
    - month (int, 1-12, compute from date)
    - year (int e.g. compute from date)
    
    User instruction: {instruction}
    
    Requirements for the code:
    
    1. Assume the Dataframe is already loaded as 'df'
    2. Use matplotlib for plotting. 
    3. Add clear title, axis labels, and legend if needed
    3. Save the figure as '{out_path_v1}'
    4. Do not call plt.show()
    6. Close all plots with plt.close().
    7. Add all necessary import python statements
    8. CRITICAL: 'date' is datetime64 - never use string concatenation on it
       Filter by year/quarter using the computed integer columns from earlier. 
    9. Do NOT use ax.bar_label/plt.bar_label. If you want value labels, use ax.text with f-strings
    10. Only use plain format strings such as f"{{v:,.0f}}" (no spaces inside the format specifier)
    11. Stick to basic, widely supported functions (bar, plot, legend, title, xlabel, ylabel, tight_layout, savefig).
    
    Return ONLY the code wrapped in the <execute_python> tags
    """
    
    response = utils.get_response(model, prompt)
    return response

In [4]:
instruction = "Create a plot comparing Q1 coffee sales in 2024 vs 2025 using the data in coffee_sales.csv"
model="gpt-4o-mini"
out_path_v1="chart_v1.png"
code_v1 = generate_chart_code(instruction, model, out_path_v1)
utils.print_html(code_v1, title="LLM output with first draft code")


AuthenticationError: Error code: 401 - Incorrect API key provided: sk-rrjsz***************************************FoOE. You can find your API key at https://platform.openai.com/account/api-keys.

In [ ]:
match = re.search(r"<execute_python>([\s\S]*?)</execute_python>")
if match:
    initial_code = match.group(1).strip()
    utils.print_html(initial_code, title="Extracted code to execute")
    utils.run_generated_code(initial_code, df, model="gpt-40-mini")

utils.print_html(
    content="chart_v1.png",
    title="Generated chart (v1)",
    is_image=True
)

In [ ]:
def reflect_on_image_and_regenerate(
    chart_path: str,
    instruction: str,
    model_name: str,
    out_path_v2: str,
    code_v1: str 
    ) -> tuple[str, str]:
    """
      Critique the chart IMAGE (in the img path) and the original code against the instruction,
      then return a refined matplotlib code.
      Returns (feedback, refined_code_with_tags). Supports 
    """
    media_type, b64 = utils.encode_image_b64(chart_path)
    prompt = f"""
      You are a data visualization expert.
      
      Your task: critique the attached chart and the original code against the given instruction,
      then return the improved matplotlib code. 
      
      Original code (for context):
      {code_v1}
      
      OUTPUT FORMAT (STRICT):
      1) First line: a valid JSON object ONLY the "feedback" field.
      Example: {{"feedback": "The legend is unclear and the axis labels overlap." }}
      
      2) After a newline, output ONLY the refined python code wrapped in: 
        <execute_python>
        ...
        </execute_python>
      
      3) Import all necessary libraries in the code. Don't assume any imports from the original code.
      
      HARD CONSTRAINTS:
      - Do not include Markdown, backticks, or any extra prose outside the two parts above. 
      - Use pandas/matplotlib only (no seaborn).
      - Assume df already exists; do not read from files.
      - Always call plt.close() at the end (no plt.show()).
      - Include all necessary import statements.
      - Do NOT use ax.bar_label / plt.bar_label; If you want value labels, use ax.text with f-strings
      - Only use plain format strings such as f"{{v: ,.0f}}" (no spaces inside the format specified)
      - Stick to basic, widely supported matplotlib functions
      
      IMPORTANT: The 'date' column is already a pandas datetime64 type. 
      - Do not concatenate 'date' with 'time' using string operations.
      - To filter by year/quarter, use: df[df['year'] == 2024]
      - The 'quarter' and 'year' columns already exist as integers; use them directly.
      
      Schema (columns available in df):
      - date (datetime64 - already parsed; use df['date'].dt.year, etc)
      - time (string, HH:MM - do NOT concatenate with date)
      - cash_type (string: 'card' or 'cash')
      - card (string)
      - price (float)
      - coffee_name (string)
      - quarter (int, 1-4)
      - month (int, 1-12)
      - year (int)
      
      CRITICAL TYPE RULE: 'date' is already datetime64. 
      - NEVER do: df['date'] + ' ' + df['time'] -> this will crash.
      - ALWAYS filter by year/quarter using the integer columns: df[df['year'] == 2024]
      
      Instruction:
      {instruction}
    """
    
    lower = model_name.lower()
    if "claude" in lower or "anthropic" in lower:
      # TODO 
      image_resolve_args = [model_name, prompt, media_type, b64]
      content = utils.image_anthropic_call(*image_resolve_args)
    else:
      content = utils.image_openai_call(model_name, prompt, media_type, b64)
      
    lines = content.strip().splitlines()
    json_line = lines[0].strip() if lines else ''
    try:
      obj = json.loads(json_line)
    except Exception as e:
       # Fallback: try to capture the first {...} in all the content
      m_json = re.search(r"\{.*?\}", content, flags=re.DOTALL)
      if m_json:
        try:
          obj = json.loads(m_json.group(0))
        except Exception as e2:
          obj = {"feedback": f"Failed to parse JSON: {e2}", "refined_code": ""}
      else:
        obj = { "feedback": f"Failed to find JSON: {e}", "refined_code": ""}
    
    m_code = re.search(r"<execute_python>([\s\S]*?)</execute_python>", content)
    refined_code_body = m_code.group(1).strip() if m_code else ""
    # TODO
    refined_code = utils.ensure_execute_python(refined_code_body)
    feedback = str(obj.get("feedback", "")).strip()
    return feedback, refined_code
        